In [ ]:
# PURPOSE OF NOTEBOOK

# clean data
# standardize data
# deal with deduplicate data
# save parquet


# Section 1 — Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import re
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,BooleanType,DateType,DecimalType


import warnings
warnings.filterwarnings("ignore")

# Section 2 — Spark Session

In [2]:

try:
    spark.stop()
except:
    pass
from pyspark.sql import SparkSession 

spark = (
    SparkSession.builder
    .appName("S3App")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 07:00:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/26 07:00:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
spark

# SECTION 3 — File Paths

In [4]:
# # path variables for data files

# Ball_By_Ball_file = "/opt/spark-data/ipl_data/1-raw/Ball_By_Ball.csv"
# Match_file= "/opt/spark-data/ipl_data/1-raw/Match.csv"
# Player_file= "/opt/spark-data/ipl_data/1-raw/Player.csv"
# Player_match_file= "/opt/spark-data/ipl_data/1-raw/Player_match.csv"
# Team_file= "/opt/spark-data/ipl_data/1-raw/Team.csv"


# Base path for raw data
BASE_PATH = "/opt/spark-data/1-raw"

# File paths
Ball_By_Ball_file = f"{BASE_PATH}/Ball_By_Ball.csv"
Match_file = f"{BASE_PATH}/Match.csv"
Player_file = f"{BASE_PATH}/Player.csv"
Player_match_file = f"{BASE_PATH}/Player_match.csv"
Team_file = f"{BASE_PATH}/Team.csv"

# Section 4 — Schemas

In [5]:
ball_by_ball_schema = StructType([
    StructField("match_id", IntegerType(), True),
    StructField("over_id", IntegerType(), True),
    StructField("ball_id", IntegerType(), True),
    StructField("innings_no", IntegerType(), True),
    StructField("team_batting", StringType(), True),
    StructField("team_bowling", StringType(), True),
    StructField("striker_batting_position", IntegerType(), True),
    StructField("extra_type", StringType(), True),
    StructField("runs_scored", IntegerType(), True),
    StructField("extra_runs", IntegerType(), True),
    StructField("wides", IntegerType(), True),
    StructField("legbyes", IntegerType(), True),
    StructField("byes", IntegerType(), True),
    StructField("noballs", IntegerType(), True),
    StructField("penalty", IntegerType(), True),
    StructField("bowler_extras", IntegerType(), True),
    StructField("out_type", StringType(), True),
    StructField("caught", BooleanType(), True),
    StructField("bowled", BooleanType(), True),
    StructField("run_out", BooleanType(), True),
    StructField("lbw", BooleanType(), True),
    StructField("retired_hurt", BooleanType(), True),
    StructField("stumped", BooleanType(), True),
    StructField("caught_and_bowled", BooleanType(), True),
    StructField("hit_wicket", BooleanType(), True),
    StructField("obstructingfeild", BooleanType(), True),
    StructField("bowler_wicket", BooleanType(), True),
    StructField("match_date", StringType(), True),
    StructField("season", IntegerType(), True),
    StructField("striker", IntegerType(), True),
    StructField("non_striker", IntegerType(), True),
    StructField("bowler", IntegerType(), True),
    StructField("player_out", IntegerType(), True),
    StructField("fielders", IntegerType(), True),
    StructField("striker_match_sk", IntegerType(), True),
    StructField("strikersk", IntegerType(), True),
    StructField("nonstriker_match_sk", IntegerType(), True),
    StructField("nonstriker_sk", IntegerType(), True),
    StructField("fielder_match_sk", IntegerType(), True),
    StructField("fielder_sk", IntegerType(), True),
    StructField("bowler_match_sk", IntegerType(), True),
    StructField("bowler_sk", IntegerType(), True),
    StructField("playerout_match_sk", IntegerType(), True),
    StructField("battingteam_sk", IntegerType(), True),
    StructField("bowlingteam_sk", IntegerType(), True),
    StructField("keeper_catch", BooleanType(), True),
    StructField("player_out_sk", IntegerType(), True),
    StructField("matchdatesk", DateType(), True)
])


# creating schema variable manually
match_schema = StructType([
    StructField("match_sk", IntegerType(), True),
    StructField("match_id", IntegerType(), True),
    StructField("team1", StringType(), True),
    StructField("team2", StringType(), True),
    StructField("match_date", StringType(), True), # here correction required to read as string then convert date type
    StructField("season_year", IntegerType(), True),
    StructField("venue_name", StringType(), True),
    StructField("city_name", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("toss_winner", StringType(), True),
    StructField("match_winner", StringType(), True),
    StructField("toss_name", StringType(), True),
    StructField("win_type", StringType(), True),
    StructField("outcome_type", StringType(), True),
    StructField("manofmach", StringType(), True),
    StructField("win_margin", IntegerType(), True),
    StructField("country_id", IntegerType(), True)
])

# creating schema variable manually
player_schema = StructType([
    StructField("player_sk", IntegerType(), True),
    StructField("player_id", IntegerType(), True),
    StructField("player_name", StringType(), True),
    StructField("dob", DateType(), True),
    StructField("batting_hand", StringType(), True),
    StructField("bowling_skill", StringType(), True),
    StructField("country_name", StringType(), True)
])




# creating schema variable manually
player_match_schema = StructType([
    StructField("player_match_sk", IntegerType(), True),
    StructField("playermatch_key", DecimalType(), True),
    StructField("match_id", IntegerType(), True),
    StructField("player_id", IntegerType(), True),
    StructField("player_name", StringType(), True),
    StructField("dob", DateType(), True),
    StructField("batting_hand", StringType(), True),
    StructField("bowling_skill", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("role_desc", StringType(), True),
    StructField("player_team", StringType(), True),
    StructField("opposit_team", StringType(), True),
    StructField("season_year", IntegerType(), True),
    StructField("is_manofthematch", BooleanType(), True),
    StructField("age_as_on_match", IntegerType(), True),
    StructField("isplayers_team_won", BooleanType(), True),
    StructField("batting_status", StringType(), True),
    StructField("bowling_status", StringType(), True),
    StructField("player_captain", StringType(), True),
    StructField("opposit_captain", StringType(), True),
    StructField("player_keeper", StringType(), True),
    StructField("opposit_keeper", StringType(), True)
])

# creating schema variable manually
team_schema = StructType([
    StructField("team_sk", IntegerType(), True),
    StructField("team_id", IntegerType(), True),
    StructField("team_name", StringType(), True)
])








# Section 5 — Read CSV Files for transformations

In [6]:
# loading data from match.csv
ball_by_ball_df = spark.read.schema(ball_by_ball_schema).format("csv").option("header","true").load(Ball_By_Ball_file)

match_df = spark.read.schema(match_schema).format("csv").option("header","true").load(Match_file)

player_df = spark.read.schema(player_schema).format("csv").option("header","true").load(Player_file)

player_match_df = spark.read.schema(player_match_schema).format("csv").option("header","true").load(Player_match_file)

team_df = spark.read.schema(team_schema).format("csv").option("header","true").load(Team_file)

# SECTION__6 Check Row Counts


In [7]:
print("Ball By Ball Rows:", ball_by_ball_df.count())

print("Match Rows:", match_df.count())

print("Player Rows:", player_df.count())

print("Player Match Rows:", player_match_df.count())

print("Team Rows:", team_df.count())

Ball By Ball Rows: 150451
Match Rows: 637
Player Rows: 497
Player Match Rows: 13993
Team Rows: 13


In [8]:
print(team_df.columns)

['team_sk', 'team_id', 'team_name']


# SECTION 7 — Standardize Column Names

In [9]:
import re

def clean_column_names(df):

    for old_col in df.columns:

        new_col = (
            old_col.strip()
            .lower()
        )

        new_col = re.sub(r'[^a-zA-Z0-9]+', '_', new_col)

        new_col = re.sub(r'_+', '_', new_col)

        new_col = new_col.strip('_')

        df = df.withColumnRenamed(old_col, new_col)

    return df

#  Apply cleaning

In [10]:
#  Apply cleaning

ball_by_ball_df = clean_column_names(ball_by_ball_df)

match_df = clean_column_names(match_df)

player_df = clean_column_names(player_df)

player_match_df = clean_column_names(player_match_df)

team_df = clean_column_names(team_df)

In [11]:
print(ball_by_ball_df.columns)

print(match_df.columns)

print(player_df.columns)

print(player_match_df.columns)

print(team_df.columns)

['match_id', 'over_id', 'ball_id', 'innings_no', 'team_batting', 'team_bowling', 'striker_batting_position', 'extra_type', 'runs_scored', 'extra_runs', 'wides', 'legbyes', 'byes', 'noballs', 'penalty', 'bowler_extras', 'out_type', 'caught', 'bowled', 'run_out', 'lbw', 'retired_hurt', 'stumped', 'caught_and_bowled', 'hit_wicket', 'obstructingfeild', 'bowler_wicket', 'match_date', 'season', 'striker', 'non_striker', 'bowler', 'player_out', 'fielders', 'striker_match_sk', 'strikersk', 'nonstriker_match_sk', 'nonstriker_sk', 'fielder_match_sk', 'fielder_sk', 'bowler_match_sk', 'bowler_sk', 'playerout_match_sk', 'battingteam_sk', 'bowlingteam_sk', 'keeper_catch', 'player_out_sk', 'matchdatesk']
['match_sk', 'match_id', 'team1', 'team2', 'match_date', 'season_year', 'venue_name', 'city_name', 'country_name', 'toss_winner', 'match_winner', 'toss_name', 'win_type', 'outcome_type', 'manofmach', 'win_margin', 'country_id']
['player_sk', 'player_id', 'player_name', 'dob', 'batting_hand', 'bowling

# SECTION 9 — Remove Duplicates

In [12]:
ball_by_ball_df = ball_by_ball_df.dropDuplicates()

match_df = match_df.dropDuplicates()

player_df = player_df.dropDuplicates()

player_match_df = player_match_df.dropDuplicates()

team_df = team_df.dropDuplicates()

# SECTION 9.1__Check Row Counts

# 9.0 Date column corrections ---Match_df need to transform -Match_Date column DataType Corrections Analysis-

In [13]:
match_df.select("match_date").show(5)

+----------+
|match_date|
+----------+
| 4/15/2011|
|04-09-2012|
| 4/25/2014|
|05-04-2015|
| 4/28/2016|
+----------+
only showing top 5 rows



In [14]:
# match df -- convert date type as per data present like M/d/yyyy in csv file 

from pyspark.sql.functions import to_date, col

match_df = match_df.withColumn(
    "match_date",
    to_date(col("match_date"), "M/d/yyyy")
)


# ball_by_ball_df -- convert date type as per data present like M/d/yyyy in csv file 

from pyspark.sql.functions import to_date, col

ball_by_ball_df = ball_by_ball_df.withColumn(
    "match_date",
    to_date(col("match_date"), "M/d/yyyy")
)



In [15]:
print(match_df.select("match_date").show(5))
# print(match_df.show())

+----------+
|match_date|
+----------+
|2011-04-15|
|      NULL|
|2014-04-25|
|      NULL|
|2016-04-28|
+----------+
only showing top 5 rows

None


In [16]:
print(ball_by_ball_df.select("match_date").show(5))

26/06/26 07:00:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 21:=========>                                                (1 + 5) / 6]

+----------+
|match_date|
+----------+
|2013-04-22|
|2009-04-27|
|2009-05-11|
|2009-05-11|
|2015-04-16|
+----------+
only showing top 5 rows

None


# SECTION 10 — Basic Data Validation

# Example 1 — Invalid Innings

In [17]:
# innings cannot be less than 2
ball_by_ball_df.filter(col("innings_no") > 2).count()

81

# Example 2 - Negative runs

In [18]:
# negative runs do not exist
ball_by_ball_df.filter(col("runs_scored") < 0).count()

0

# Example 3 — Match Date Nulls

In [19]:
# match date should present each records
match_df.filter(col("match_date").isNull()).count()

235

In [20]:
# we found 637 null
match_df.select("match_date").show()
# this o/p shows null values so its need to reload in as string type datastructure and then 
# transform into date format shown in string output then read as dateType.

+----------+
|match_date|
+----------+
|2011-04-15|
|      NULL|
|2014-04-25|
|      NULL|
|2016-04-28|
|      NULL|
|2008-04-26|
|2010-03-23|
|2014-05-14|
|2016-05-19|
|      NULL|
|2009-05-24|
|2012-04-13|
|      NULL|
|2016-04-30|
|2017-04-13|
|      NULL|
|      NULL|
|2013-05-18|
|2008-04-22|
+----------+
only showing top 20 rows



# SECTION 11 — File path to Write Bronze Layer

In [21]:
# # path variables for data files writing path_to_parquet

# ball_by_ball_file_path_to_parquet = "/opt/spark-data/ipl_data/2_bronze/ball_by_ball_parquet"
# match_file_path_to_parquet= "/opt/spark-data/ipl_data/2_bronze/match_parquet"
# player_file_path_to_parquet= "/opt/spark-data/ipl_data/2_bronze/player_parquet"
# player_match_file_path_to_parquet= "/opt/spark-data/ipl_data/2_bronze/player_match_parquet"
# team_file_path_to_parquet= "/opt/spark-data/ipl_data/2_bronze/team_parquet"

# ==========================
# Bronze Layer
# ==========================
BRONZE_PATH = "/opt/spark-data/2_bronze"

ball_by_ball_file_path_to_parquet = f"{BRONZE_PATH}/ball_by_ball_parquet"
match_file_path_to_parquet = f"{BRONZE_PATH}/match_parquet"
player_file_path_to_parquet = f"{BRONZE_PATH}/player_parquet"
player_match_file_path_to_parquet = f"{BRONZE_PATH}/player_match_parquet"
team_file_path_to_parquet = f"{BRONZE_PATH}/team_parquet"

In [22]:
# Write to bronze parquet Dataset
ball_by_ball_df.write \
    .mode("overwrite") \
    .parquet(ball_by_ball_file_path_to_parquet)

In [23]:
# Write to bronze parquet Dataset
match_df.write.mode("overwrite").parquet(match_file_path_to_parquet)

In [24]:
# Write to bronze parquet Dataset
player_df.write \
    .mode("overwrite") \
    .parquet(player_file_path_to_parquet)

In [25]:
# Write to bronze parquet Dataset
player_match_df.write \
    .mode("overwrite") \
    .parquet(player_match_file_path_to_parquet)

In [26]:
# Write to bronze parquet Dataset
team_df.write \
    .mode("overwrite") \
    .parquet(team_file_path_to_parquet)

# SECTION 12 — Verify Written Data

In [27]:
bronze_ball_df = spark.read.parquet(match_file_path_to_parquet)

bronze_ball_df.show(5)

+--------+--------+-------------------+--------------------+----------+-----------+--------------------+-------------+------------+--------------------+--------------------+---------+--------+------------+---------+----------+----------+
|match_sk|match_id|              team1|               team2|match_date|season_year|          venue_name|    city_name|country_name|         toss_winner|        match_winner|toss_name|win_type|outcome_type|manofmach|win_margin|country_id|
+--------+--------+-------------------+--------------------+----------+-----------+--------------------+-------------+------------+--------------------+--------------------+---------+--------+------------+---------+----------+----------+
|     186|  501214|   Rajasthan Royals|Kolkata Knight Ri...|2011-04-15|       2011|Sawai Mansingh St...|       Jaipur|       India|Kolkata Knight Ri...|Kolkata Knight Ri...|    field| wickets|      Result|G Gambhir|         9|         1|
|     256|  548319|    Deccan Chargers|      Mum

In [28]:
ls

01_ipl_data_ingestion.ipynb
02_ipl_data_validate_write_bronze.ipynb*
03_ipl_data_Transform_silver.ipynb*
04_ipl_data_Transformation_gold.ipynb*
05_spark_sql_analytic.ipynb
